# LLiMa Introduction

**LLiMa** is SiMa's toolchain for generative models on Modalix — LLMs (text), VLMs (image + text),
and ASR (speech-to-text). This notebook covers what LLiMa is, the model families it ships, how to
read a model ID, and every `llima` CLI subcommand.

The distinction to get right before anything else: the **LLiMa CLI** *prepares* generative models,
and the **`pyneat.genai` API** *runs* them from your application. Everything in this folder follows
from that split.

The cheap commands (`--help`, `search`, `list`) appear inline as text. The heavy ones
(`pull`, `run`, `benchmark-server`) are kept as printed strings so no cell can fire one by accident —
run those on the DevKit yourself.

## What LLiMa is

LLiMa owns three jobs:

1. **Model preparation / download** — fetch a model already quantized and compiled for Modalix from a
   remote catalog, and lay it down as a *model directory* on the board.
2. **Command-line testing** — a quick interactive `run` to sanity-check a model without writing code.
3. **Benchmarking** — `benchmark-server` starts an HTTP server used to measure a model's throughput.

The CLI is a single binary, `llima`, a Python entry point (`sima_lmm.devkit.devkit_demo`).

Its key output is a **model directory** under `/media/nvme/llima/models/<model-id>`. That directory
is the hand-off point: LLiMa produces it, and the `pyneat.genai` API consumes it.

> Source of truth: `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`
> ("LLiMa owns GenAI model preparation, command-line testing, and benchmarking. Neat owns the
> application-facing API and runtime integration once the model is used inside your app.")

## Supported model families

`llima search` returns a catalog of models already prepared for Modalix. They fall into three
families. The cell below embeds that catalog as text and counts it.

In [ ]:
# The `llima search` catalog, embedded as text. This cell only counts strings.
CATALOG = """
llava-1.5-7b-hf-a16w4
Llama-3.1-8B-Instruct-a16w4
Llama-3.2-3B-Instruct-a16w4
Phi-3.5-mini-instruct-a16w4
paligemma-3b-pt-224-a16w8
gemma-3-4b-it-a16w4
Llama-2-7b-chat-hf-a16w4
whisper-small-a16w8
gemma-3-1b-it-a16w4
Mistral-7B-Instruct-v0.3-a16w4
gemma3-siglip448-a16w4
Qwen2.5-0.5B-Instruct-GPTQ-a16w4
Qwen2.5-7B-Instruct-GPTQ-a16w4
Qwen3-0.6B-GPTQ-a16w4
Qwen3-1.7B-GPTQ-a16w4
Qwen3-4B-Instruct-2507-GPTQ-a16w4
Qwen3-8B-GPTQ-a16w4
LFM2-VL-450M-a16w4
LFM2-VL-1.6B-a16w4
LFM2-VL-3B-a16w4
Qwen2.5-VL-3B-Instruct-GPTQ-a16w4
Qwen3-VL-4B-Instruct-GPTQ-a16w4
Qwen3-VL-8B-Instruct-GPTQ-a16w4
LFM2-350M-a16w4
LFM2-1.2B-a16w4
LFM2-2.6B-a16w4
Qwen2.5-VL-7B-Instruct-GPTQ-a16w4
Qwen3-VL-2B-Instruct-GPTQ-a16w4
gemma-4-E2B-it-GPTQ-a16w4
gemma-4-E4B-it-GPTQ-a16w4
LFM2.5-230M-a16w4
LFM2.5-350M-a16w4
LFM2.5-1.2B-Instruct-a16w4
LFM2.5-1.2B-Thinking-a16w4
LFM2.5-VL-450M-a16w4
LFM2.5-VL-1.6B-a16w4
""".split()

vlm = [m for m in CATALOG if "-VL-" in m or m.startswith(("llava", "paligemma", "gemma3-siglip"))]
asr = [m for m in CATALOG if m.startswith("whisper")]
llm = [m for m in CATALOG if m not in vlm and m not in asr]

print(f"catalog size: {len(CATALOG)} models")
print(f"  LLM (text):        {len(llm)}  e.g. {llm[:3]}")
print(f"  VLM (image+text):  {len(vlm)}  e.g. {vlm[:3]}")
print(f"  ASR (speech->text):{len(asr):>3}  e.g. {asr}")

**Interpretation.** The catalog is dominated by decoder LLMs (Qwen3, Llama, Mistral, Phi,
Gemma, LFM2), a strong set of VLMs (the `*-VL-*` families plus llava / paligemma / siglip), and a
single ASR model (`whisper-small-a16w8`). Family cheat-sheet:

- **LLM** — text in, text out. Chat, summarization, tool-calling.
- **VLM** — image(s) + text in, text out. Captioning, visual Q&A, document reading.
- **ASR** — audio in, text out. Transcription.

Every model ID ends in a **quantization suffix** — the next section decodes it.

## Quantization suffixes (a correctness trap)

The suffix after the model name is the LLiMa quantization scheme. It is not decoration — it tells you
the weight precision, which drives the on-disk size and the memory footprint.

- **`a16w4`** = **16-bit activations / 4-bit weights** (the common case; smallest weights).
- **`a16w8`** = **16-bit activations / 8-bit weights** (e.g. `whisper-small-a16w8`,
  `paligemma-3b-pt-224-a16w8`).

`a16w8` weights are roughly twice the size of `a16w4` weights for the same parameter count, all else
equal.

In [ ]:
def decode_suffix(model_id: str) -> str:
    if model_id.endswith("a16w4"):
        return "16-bit activations / 4-bit weights"
    if model_id.endswith("a16w8"):
        return "16-bit activations / 8-bit weights"
    return "unknown quantization"

for m in ["Qwen3-4B-Instruct-2507-GPTQ-a16w4", "whisper-small-a16w8"]:
    print(f"{m:45s} -> {decode_suffix(m)}")

**Interpretation.** `Qwen3-4B-...-a16w4` keeps its 4B weights at 4 bits each; `whisper-small`
keeps its weights at 8 bits. When you compare two models, read the suffix before you read the
parameter count — an `a16w8` model of half the parameters can still be as heavy on disk as an
`a16w4` model with more parameters.

## The CLI: top-level surface

`llima --help`:

```text
usage: llima [-h] {run,search,pull,list,rm,benchmark-server} ...

Llima unified CLI

positional arguments:
  {run,search,pull,list,rm,benchmark-server}
    run                 Run a model
    search              Search remote models
    pull                Download a model
    list                List local models
    rm                  Remove a local model
    benchmark-server    Start a server for benchmarking a model
```

**Correctness trap #1:** the serve subcommand is **`benchmark-server`, NOT `serve`**. There is no
`llima serve`. Typing `llima serve` will error.

In [ ]:
# A map of the six subcommands. Prints only.
SUBCOMMANDS = {
    "search":           ("cheap", "Search the remote catalog"),
    "list":             ("cheap", "List models already on the board"),
    "pull":             ("heavy", "Download a model directory (gigabytes)"),
    "run":              ("heavy", "Load and run a model (LLM/VLM/ASR)"),
    "rm":               ("cheap", "Delete a local model directory"),
    "benchmark-server": ("heavy", "Start an HTTP benchmarking server (NOT `serve`)"),
}
for name, (cost, desc) in SUBCOMMANDS.items():
    print(f"llima {name:16s} [{cost:5s}] {desc}")

## `llima search` — browse the remote catalog (cheap)

`llima search --help`:

```text
usage: llima search [-h] [term]

positional arguments:
  term        Search term (optional)
```

With no term it lists the full remote catalog; with a term it filters. **On disk: nothing.** It only
queries the remote catalog and prints IDs. Run it before `pull` to get the exact model ID string.

```bash
llima search              # full catalog
llima search Qwen3        # filter to the Qwen3 family
```

Filtering locally against the embedded catalog:

In [ ]:
term = "Qwen3"
print(f"llima search {term}  ->")
for m in CATALOG:
    if term.lower() in m.lower():
        print(" ", m)

## `llima list` — what is already on the board (cheap)

**On disk: nothing changes.** It reads the local model directory and prints the model IDs present.

```bash
llima list
```

Run it before `llima run`, or before any notebook in `../02-run-llm-vlm/`, to confirm the model you
intend to use is present and to copy its exact ID.

## `llima pull` — download a model (heavy)

`llima pull --help`:

```text
usage: llima pull [-h] model

positional arguments:
  model       Model ID
```

**What happens on disk:** `pull` downloads the model (weights already quantized/compiled for Modalix)
and writes a **model directory** under `/media/nvme/llima/models/<model-id>`. That directory is what
`pyneat.genai` and `llima run` later load.

> A pull moves several GB and can take a few minutes. Run it on the DevKit when you are ready.

In [ ]:
# Reference strings only - printing them, not pulling anything.
PULL_LLM = "llima pull Qwen3-4B-Instruct-2507-GPTQ-a16w4"
PULL_SMALL_LLM = "llima pull LFM2.5-230M-a16w4"     # a small LLM
PULL_SMALL_VLM = "llima pull LFM2-VL-450M-a16w4"    # a small VLM

for label, cmd in [("LLM", PULL_LLM), ("small LLM", PULL_SMALL_LLM), ("small VLM", PULL_SMALL_VLM)]:
    print(f"{label:11s}: {cmd}")
print("lands at   : /media/nvme/llima/models/<model-id>/")

## `llima run` — quick interactive test (heavy)

`llima run --help`:

```text
usage: llima run [-h] [--mode {cli,web}] [--stt_model_path STT_MODEL_PATH]
                 [--system_prompt SYSTEM_PROMPT | --system_prompt_file ... |
                  --chat_template ... | --chat_template_file ...]
                 [--parallel_load | --no-parallel_load] [--log_level LOG_LEVEL]
                 model

positional arguments:
  model                 Model path or model ID
options:
  --mode {cli,web}
  --stt_model_path STT_MODEL_PATH   Path to the elf files for speech-to-text model.
  --system_prompt / --system_prompt_file / --chat_template / --chat_template_file
  --parallel_load, --no-parallel_load
  --log_level LOG_LEVEL
```

`llima run` loads a model directory into memory and gives you a no-code smoke test. **On disk:**
nothing new is written.

- **`--mode cli`** (default) gives you an interactive prompt in the terminal.
- **`--mode web`** starts an HTTP server on `0.0.0.0:9998` instead. Despite the name it serves no
  browser page — it registers POST endpoints only (an OpenAI-compatible surface such as
  `/v1/chat/completions`, plus `/api/chat` and `/api/generate` aliases). Drive it with `curl` or an
  OpenAI-style client, not a browser tab. Note that `9998` is also the default port of
  `pyneat.genai.GenAIServer`, so the two cannot run at once on one board.

**Exiting a run:** press **Ctrl+C**. There is no `exit` or `quit` command, and Ctrl+D is not handled.
Ctrl+C is caught deliberately, prints `User interrupt detected. Exiting...`, and then unloads the
model and disconnects the MLA dispatcher. Let that teardown finish rather than killing the process,
or you can leave a stale lock behind.

**Correctness trap #2 — ASR.** A speech-to-text model is run by passing its ELF directory explicitly
with **`--stt_model_path <elf-dir>`**. Text/vision models do not need this flag; ASR does.

In [ ]:
# Reference strings only - printing the commands, not running them.
# LLM smoke test (text):
RUN_LLM = "llima run --mode cli Qwen3-4B-Instruct-2507-GPTQ-a16w4"

# LLM with a system prompt:
RUN_LLM_SYS = ('llima run --mode cli '
               '--system_prompt \'You are concise.\' '
               'Qwen3-4B-Instruct-2507-GPTQ-a16w4')

# ASR smoke test (audio) - note the --stt_model_path <elf-dir> (trap #2):
RUN_ASR = ("llima run --stt_model_path "
           "/media/nvme/llima/models/whisper-small-a16w8 "
           "whisper-small-a16w8")

for label, cmd in [("LLM", RUN_LLM), ("LLM+system", RUN_LLM_SYS), ("ASR", RUN_ASR)]:
    print(f"{label:11s}: {cmd}")

## `llima rm` — delete a local model (cheap)

Removes a model directory under `/media/nvme/llima/models/`, freeing disk. **On disk:** deletes that
model's directory. It does not touch the remote catalog, so you can `pull` it again later.

```bash
llima rm <model-id>
```

## `llima benchmark-server` — throughput server (heavy)

**Correctness trap #1 again:** the subcommand is `benchmark-server`, **not** `serve`.

`llima benchmark-server --help`:

```text
usage: llima benchmark-server [-h] [--port PORT] model

positional arguments:
  model        Model ID or path
options:
  --port PORT  Listening port
```

Starts an HTTP server that hosts the model for throughput measurement. **On disk:** nothing; it holds
the model in memory and serves requests. Exit it with Ctrl+C, as with `llima run`.

> Distinct from the application server `pyneat.genai.GenAIServer` (OpenAI-compatible endpoints on
> port 9998 by default), which you build into your own app — see
> `../05-genai-server/01_genai_server.ipynb`.

In [ ]:
# Reference string only - printed, never executed.
BENCH = "llima benchmark-server --port 8080 Qwen3-4B-Instruct-2507-GPTQ-a16w4"
print(BENCH)
print("# WRONG (there is no `serve` subcommand):")
print("# llima serve Qwen3-4B-Instruct-2507-GPTQ-a16w4   <-- errors")

## Recap and next

- **LLiMa (CLI)** prepares a *model directory*; **`pyneat.genai`** runs it. Do not confuse the two.
- Three families: **LLM / VLM / ASR**.
- Read the **quantization suffix** (`a16w4` vs `a16w8`) before the parameter count.
- Six subcommands: `search`, `list`, `pull`, `run`, `rm`, `benchmark-server`.
  Cheap: `search`, `list`, `rm`. Heavy: `pull`, `run`, `benchmark-server`.
- **Trap #1:** it is `benchmark-server`, not `serve`.
- **Trap #2:** ASR runs via `llima run --stt_model_path <elf-dir> <model>`.
- `pull` writes to `/media/nvme/llima/models/<id>/`; that directory is what the Python API loads next.
- Exit `llima run` and `benchmark-server` with **Ctrl+C**.

Next: `../02-run-llm-vlm/01_run_llm.ipynb` runs the LLM from Python.

## References

- CLI output: on-board `llima --help` and each subcommand's `--help`
- `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`
- `/workspace/core/tutorials/019_run_an_llm/README.md`
- Official docs: developer.sima.ai/software/getting-started/ and .../genai-llima/runtime